In [1]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import os
import requests
import joblib

import numpy
import pandas

import matplotlib.pyplot as plt
import scipy.stats as stats

In [2]:
from catboost import CatBoostRegressor 
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.linear_model import ElasticNetCV, LassoCV, RidgeCV
from sklearn.svm import SVR

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.ensemble import AdaBoostRegressor

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error

from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score


import sklearn.linear_model as linear_model
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

In [3]:
train = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/train.csv').drop(columns=['ID','efs_time'])
test = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/test.csv').drop(columns=['ID'])

In [4]:
encoder = OneHotEncoder(sparse_output=False)

In [5]:
for i in zip(train.columns,train.dtypes):
    if i[1]=='O':
        train[i[0]]=encoder.fit_transform(train[i[0]].to_numpy().reshape(-1,1))
    else:
        train[i[0]]=train[i[0]]
for i in zip(test.columns,test.dtypes):
    if i[1]=='O':
        test[i[0]]=numpy.array(encoder.fit_transform(test[i[0]].to_numpy().reshape(-1,1)))
    else:
        test[i[0]]=numpy.array(test[i[0]])

In [6]:
train = train.fillna(0)
test = test.fillna(0)

train_y = train['efs']
train_x = train.drop(columns=['efs'])

for i in test:
    test[i]=test[i].to_numpy()

In [7]:
from hyperopt import STATUS_OK, Trials, fmin, hp, tpe
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import random

X_train, X_test, y_train, y_test = train_test_split(train_x, train_y, test_size = 0.3, random_state = 0)

space={ 'n_estimators': hp.quniform("n_estimators", 1000, 20000, 1),
        'max_depth': hp.quniform("max_depth", 3, 30, 1),
        'gamma': hp.uniform ('gamma', 1,9),
        'reg_alpha' : hp.quniform('reg_alpha', 40,180,1),
        'reg_lambda' : hp.uniform('reg_lambda', 0,1),
        'colsample_bytree' : hp.uniform('colsample_bytree', 0.5,1),
        'min_child_weight' : hp.quniform('min_child_weight', 0, 10, 1),
        'subsample' : hp.uniform('subsample', 0, 1),
        'learning_rate': hp.uniform('learning_rate', 0.0000001, 1),
        'seed': 0
    }

def objective(space):
    XGB=XGBRegressor(
                    n_estimators =int(space['n_estimators']), max_depth = int(space['max_depth']), gamma = space['gamma'],
                    reg_alpha = int(space['reg_alpha']),min_child_weight=int(space['min_child_weight']),
                    colsample_bytree=int(space['colsample_bytree']))
    
    evaluation = [( X_test, y_test)]
    
    XGB.fit(X_train, y_train,
            eval_set=evaluation, eval_metric="auc",
            early_stopping_rounds=10,verbose=False)
    

    pred = XGB.predict(X_test)
    accuracy = accuracy_score(y_test, pred>0.5)
    print ("SCORE:", accuracy)
    return {'loss': -accuracy, 'status': STATUS_OK }

trials = Trials()

XGB_hyperparams = fmin(fn=objective,
                space=space,
                max_evals=2400,
                algo=tpe.suggest)

XGB_hyperparams

In [8]:
from sklearn.model_selection import RandomizedSearchCV
import scipy.stats.distributions as dists

space={ 'n_estimators': numpy.arange(1000, 20000),
        'max_depth': numpy.arange(3, 30),
        'gamma': numpy.arange(1,9),
        'reg_alpha' : numpy.arange(40,180),
        'reg_lambda' : dists.uniform(0, 1),
        'colsample_bytree' : dists.uniform(0.5, 1),
        'min_child_weight' : numpy.arange(0, 10),
        'subsample' : dists.uniform(0, 1),
        'learning_rate': dists.uniform(0, 1),
    }
print(space['n_estimators'])

XGB=XGBRegressor()

tree_cv = RandomizedSearchCV(XGB, param_distributions=space, cv=10, verbose=3)
tree_cv.fit(X_train, y_train)

XGB_hyperparams=tree_cv.best_params_
print("Tuned Decision Tree Parameters: {}".format(tree_cv.best_params_))
print("Best score is {}".format(tree_cv.best_score_))

[ 1000  1001  1002 ... 19997 19998 19999]
Fitting 10 folds for each of 10 candidates, totalling 100 fits
[CV 1/10] END colsample_bytree=0.7887395562361499, gamma=5, learning_rate=0.1625296508379882, max_depth=6, min_child_weight=9, n_estimators=19625, reg_alpha=177, reg_lambda=0.7374475690679785, subsample=0.5316859481296065;, score=0.074 total time=  19.5s
[CV 2/10] END colsample_bytree=0.7887395562361499, gamma=5, learning_rate=0.1625296508379882, max_depth=6, min_child_weight=9, n_estimators=19625, reg_alpha=177, reg_lambda=0.7374475690679785, subsample=0.5316859481296065;, score=0.068 total time=  17.6s
[CV 3/10] END colsample_bytree=0.7887395562361499, gamma=5, learning_rate=0.1625296508379882, max_depth=6, min_child_weight=9, n_estimators=19625, reg_alpha=177, reg_lambda=0.7374475690679785, subsample=0.5316859481296065;, score=0.069 total time=  19.3s
[CV 4/10] END colsample_bytree=0.7887395562361499, gamma=5, learning_rate=0.1625296508379882, max_depth=6, min_child_weight=9, n_e

/opt/conda/lib/python3.10/site-packages/sklearn/model_selection/_validation.py:378: FitFailedWarning: 
30 fits failed out of a total of 100.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
10 fits failed with the following error:
Traceback (most recent call last):
  File "/opt/conda/lib/python3.10/site-packages/sklearn/model_selection/_validation.py", line 686, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/opt/conda/lib/python3.10/site-packages/xgboost/core.py", line 730, in inner_f
    return func(**kwargs)
  File "/opt/conda/lib/python3.10/site-packages/xgboost/sklearn.py", line 1090, in fit
    self._Booster = train(
  File "/opt/conda/lib/python3.10/site-packages/xgboost/core.py", line 730, in inner_f
 

Tuned Decision Tree Parameters: {'colsample_bytree': 0.5178106501985065, 'gamma': 2, 'learning_rate': 0.305400259068883, 'max_depth': 25, 'min_child_weight': 5, 'n_estimators': 9027, 'reg_alpha': 74, 'reg_lambda': 0.6888659182589428, 'subsample': 0.7348719115043433}
Best score is 0.11058026315184062


In [9]:
{'colsample_bytree': 0.8048539697942243, 'gamma': 6, 'learning_rate': 0.7238195160468303, 'max_depth': 18, 'min_child_weight': 5, 
 'n_estimators': 19407, 'reg_alpha': 143, 'reg_lambda': 0.4163177870975998, 'subsample': 0.9632743217349022}

{'colsample_bytree': 0.8048539697942243,
 'gamma': 6,
 'learning_rate': 0.7238195160468303,
 'max_depth': 18,
 'min_child_weight': 5,
 'n_estimators': 19407,
 'reg_alpha': 143,
 'reg_lambda': 0.4163177870975998,
 'subsample': 0.9632743217349022}

In [10]:
svr = SVR(kernel='rbf', 
          C=1, 
          epsilon=0.1, 
          gamma=0.1)

In [11]:
gbr = GradientBoostingRegressor(n_estimators=3000, 
                                learning_rate=0.05, 
                                max_depth=4, 
                                max_features='sqrt', 
                                min_samples_leaf=15, 
                                min_samples_split=10, 
                                loss='huber', 
                                random_state =42)    

lightgbm = LGBMRegressor(colsample_bytree=0.9119276182779381, 
                         drop_rate=0.2870902369968208,
                         learning_rate=0.09797055466390917, 
                         max_bin=29, 
                         max_depth=3,
                         max_drop=111, 
                         min_child_samples=33, 
                         min_data_in_leaf=68,
                         n_estimators=277, 
                         num_leaves=25, 
                         reg_alpha=0.12688048100404745,
                         reg_lambda=0.889850076976295, 
                         skip_drop=0.5462032763564192,
                         verbosity=-1)

xgboost = XGBRegressor(n_estimators=int(XGB_hyperparams['n_estimators']),  
                       learning_rate=XGB_hyperparams['learning_rate'],
                       colsample_bytree=XGB_hyperparams['colsample_bytree'], 
                       subsample=XGB_hyperparams['subsample'], 
                       objective='reg:logistic',
                       min_child_weight=XGB_hyperparams['min_child_weight'],
                       gamma=XGB_hyperparams['gamma'],
                       max_depth=int(XGB_hyperparams['max_depth']),
                       reg_alpha=XGB_hyperparams['reg_alpha'],
                       reg_lambda=XGB_hyperparams['reg_lambda'])
#binary:logitraw
#binary:logistic

catboost = CatBoostRegressor(learning_rate=0.009,
                             depth=5,
                             l2_leaf_reg=3.5,
                             min_child_samples=32,
                             grow_policy='Depthwise',
                             iterations=8000,
                             eval_metric='RMSE',
                             od_type='Iter',
                             od_wait=50,
                             random_state=42,
                             logging_level='Silent')

In [12]:
#model = StackingRegressor(estimators=[('catboost', catboost), ('gbr', gbr),
#                                      ('xgboost', xgboost), ('lightgbm', lightgbm)],
#                                      final_estimator=svr)
model=xgboost

In [13]:
model.fit(train_x, train_y)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.5178106501985065, device=None,
             early_stopping_rounds=None, enable_categorical=False,
             eval_metric=None, feature_types=None, gamma=2, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.305400259068883, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=25, max_leaves=None,
             min_child_weight=5, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=9027, n_jobs=None,
             num_parallel_tree=None, objective='reg:logistic', ...)

In [14]:
id = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/test.csv')['ID']

In [15]:
test_predictions = model.predict(test)

In [16]:
test_predictions

array([0.1609705 , 0.7252365 , 0.19232412], dtype=float32)

In [17]:
submission = pandas.DataFrame({
    'ID': id.values,
    'prediction': test_predictions,
})
submission

,ID,prediction
0,28800,0.160970
1,28801,0.725236
2,28802,0.192324


In [18]:
submission.to_csv('submission.csv', index = False)